# Notebook 04 — Closed-Loop Generation (With WQB)

**Goal**: Run the full orchestration loop:
```
Idea → Synthesis → Validate → Novelty → WQB Simulate → Qualify → Reflect → Memory
```

After each round, the system learns from failures and improves its next iteration.

**Requires**: Notebooks 01–03 completed. WQB credentials set in `.env`.

⚠️ **WQB API quota**: each expression costs one simulation. Start with `n_rounds=1`.

In [1]:
import asyncio
import nest_asyncio
nest_asyncio.apply()

from alpha_agent.knowledge.vector_store import VectorStore
from alpha_agent.knowledge.alpha_memory import AlphaMemory
from alpha_agent.data.operator_kb import OperatorKB
from alpha_agent.data.wqb_client import WQBClient
from alpha_agent.engine.orchestrator import Orchestrator
from alpha_agent.search.bandit import DirectionBandit

store  = VectorStore()
memory = AlphaMemory()
kb     = OperatorKB()

/Users/weiyanguang/vscodeprojects/WorldQuant-Brain-Alpha/myenv/lib/python3.11/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from tqdm.autonotebook import tqdm, trange


## 1. Configure the run

In [2]:
# ─── Configuration ────────────────────────────────────────────────────────
DIRECTION  = "short-term price reversal with liquidity adjustment"
DATASET    = "pv1"
UNIVERSE   = "TOP1000"
N_ROUNDS   = 2        # start small! increase after validating setup
IDEAS_PER_ROUND   = 3
VARIANTS_PER_IDEA = 2
EXPLORE_BIAS = 0.4   # 0=explore, 1=exploit
AUTO_SUBMIT  = False  # set True to auto-submit qualified alphas
DRY_RUN      = False  # True = skip WQB (for testing prompt quality)
# ──────────────────────────────────────────────────────────────────────────

## 2. (Optional) Use Bandit to auto-select direction

In [3]:
# Uncomment to let the bandit choose the direction automatically
# bandit = DirectionBandit(memory)
# DIRECTION = bandit.select()
# print(f"Bandit selected direction: {DIRECTION}")

print(f"Direction: {DIRECTION}")
print(f"Dataset:   {DATASET} / {UNIVERSE}")
print(f"Rounds:    {N_ROUNDS}")
print(f"Dry run:   {DRY_RUN}")

Direction: short-term price reversal with liquidity adjustment
Dataset:   pv1 / TOP1000
Rounds:    2
Dry run:   False


## 3. Run the orchestration loop

In [4]:
async def run_pipeline():
    async with WQBClient() as client:
        orch = Orchestrator(
            client=client,
            vector_store=store,
            alpha_memory=memory,
            operator_kb=kb,
            auto_submit=AUTO_SUBMIT,
        )
        reports = await orch.run(
            direction=DIRECTION,
            dataset=DATASET,
            universe=UNIVERSE,
            n_rounds=N_ROUNDS,
            ideas_per_round=IDEAS_PER_ROUND,
            variants_per_idea=VARIANTS_PER_IDEA,
            explore_exploit_bias=EXPLORE_BIAS,
            dry_run=DRY_RUN,
        )
    return reports

reports = asyncio.run(run_pipeline())

───────────────────────── Round 1/2 — short-term price reversal with liquidity adjustment ─────────────────────────

Step 1 Generating 3 ideas...

Got 3 ideas

Step 2 Synthesizing 2 variants per idea...

Generated 6 expressions

...

───────────────────────── Round 2/2 — short-term price reversal with liquidity adjustment ─────────────────────────

Step 1 Generating 3 ideas...

Got 3 ideas

Step 2 Synthesizing 2 variants per idea...

KeyboardInterrupt: 

## 4. Review results

In [ ]:
import pandas as pd

summary_rows = [r.to_dict() for r in reports]
df_summary = pd.DataFrame(summary_rows)
print("Round-by-round summary:")
display(df_summary)

In [ ]:
# Check alpha memory status
stats = memory.stats()
print(f"Alpha Memory after run:")
print(f"  Total:     {stats['total']}")
print(f"  Qualified: {stats['qualified']}")
print(f"  Pass rate: {stats['pass_rate']:.1%}")

In [ ]:
# View top qualified alphas
top = memory.top_by_metric('sharpe', k=5, qualified_only=True)
print("Top 5 qualified alphas by Sharpe:")
for a in top:
    m = a['metrics']
    print(f"  sharpe={m.get('sharpe',0):.3f} fitness={m.get('fitness',0):.3f}")
    print(f"  {a['expression'][:90]}")
    print()

In [ ]:
# View recent reflections (learning)
recent = memory.recent(n=10)
failed = [a for a in recent if not a['qualified'] and a['reflection']]

print("Recent failure reflections:")
for a in failed[:3]:
    print(f"\n  Expression: {a['expression'][:60]}")
    print(f"  Reflection: {str(a['reflection'])[:300]}")

## ✅ Notebook 04 Complete

The closed loop has run. The system has:
- Generated LLM hypotheses guided by RAG context
- Synthesized and locally validated expressions
- Filtered out near-duplicates using novelty scoring
- Submitted to WQB for backtesting
- Reflected on failures and stored insights for next round

**Next**: `05_results_and_novelty_analysis.ipynb` for full analysis.